<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.2 — Ms. Womack: the Method, Then Real Stocks

Ms. Womack has savings to invest and no appetite for losing them. In M4.2 we found the best way to split her money
across five stocks. Tonight we slow down and **see** the method, then point the exact same model at real stocks.

1. **Part 1 · The idea in pictures.** What we maximize, what we hold under a limit, and what "risk" (variance) really is.
2. **Part 2 · Why covariance and not correlation.** Three experiments with Ms. Womack's own numbers.
3. **Part 3 · Ms. Womack's efficient frontier.** Beat the solver by hand, then solve at many risk limits.
4. **Part 4 · Real stocks.** Pick a data set from a menu, slide your risk tolerance, and see what to hold.

**How this notebook works.** All of the code is written. Run the setup cell, then go one cell at a time. When you
see **Predict first**, write down your guess *before* you run the next cell. Halfway through there is a contest:
beat the solver by hand. Part 4 has menus and a slider to play with.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · The idea in pictures

Here is the whole method on one page. Keep it open while you work.

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/womack_methodology.png" width="100%"/>

### The model

Let $x_k$ be the share of Ms. Womack's money in stock $k$, $\bar r_k$ its average monthly return, and
$\text{Cov}(i,j)$ the covariance between stocks $i$ and $j$.

$$\max \;\; \sum_k \bar r_k \, x_k \qquad \text{(the average return of the portfolio)}$$

subject to

$$\sum_k x_k = 1 \qquad \text{(invest all of the money)}$$

$$\sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j) \;\le\; \text{risk limit} \qquad \text{(the risk: the weighted covariance matrix, summed)}$$

$$0 \le x_k \le 1$$

In words: **make as much money as possible, without the risk going over the limit.** The return is linear (a weighted
average). The risk is not: it multiplies two decision variables together, $x_i \times x_j$. That is why this is a
nonlinear model and why we use ipopt.

### Ms. Womack's data

24 months of returns for her five stocks: one row per month, one column per stock. Each number is that month's
return as a decimal (0.05 is a 5% gain, -0.03 is a 3% loss).

In [ ]:
# Ms. Womack's monthly returns (the same numbers as in M4.2)
df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/womack_returns.csv')
df.head()

Two numbers come out of this table, and nothing else goes into the model:

- `df_return`: the **average** monthly return of each stock (the objective coefficients),
- `df_cov`: the **covariance matrix** (the risk).

In [ ]:
del df['Month']        # the data is in order; drop the month counter or it swamps the covariance matrix

df_return = df.mean()
df_cov = df.cov()

summary = pd.DataFrame({'average monthly return': df_return, 'volatility (std. dev.)': df.std()})
summary.round(4)

### Ms. Womack's model, with her numbers

Here is the model from above with Ms. Womack's actual numbers plugged in. The cell below writes it out from
`df_return` and `df_cov`, so it works for any data.

The risk is the weighted covariance grid, summed. Each stock's own variance shows up once (the diagonal, the $x_k^2$
terms), and each pair shows up **twice** (the grid has both $\text{Cov}(i,j)$ and $\text{Cov}(j,i)$), so pairs carry a 2.

In [ ]:
from IPython.display import display, Math, Markdown


def show_model(avg_returns, cov_matrix, risk_limit):
    names = list(avg_returns.index)

    # objective: one term per stock
    objective = r"\max \quad "
    for k in range(len(names)):
        term = r"%.4f\,x_{\text{%s}}" % (abs(avg_returns[names[k]]), names[k])
        if k == 0 and avg_returns[names[k]] < 0:
            objective += "-" + term
        elif k == 0:
            objective += term
        elif avg_returns[names[k]] < 0:
            objective += " - " + term
        else:
            objective += " + " + term

    # invest all of the money
    budget = ""
    for k in range(len(names)):
        if k > 0:
            budget += " + "
        budget += r"x_{\text{%s}}" % names[k]
    budget += " = 1"

    # risk, part 1: each stock with itself (the diagonal), one per line
    diagonal = r"\begin{array}{rl}"
    for i in range(len(names)):
        sign = "+"
        if i == 0:
            sign = ""
        diagonal += sign + r" & %.5f \; x_{\text{%s}}^2 \\[6pt]" % (cov_matrix.loc[names[i], names[i]], names[i])
    diagonal += r"\end{array}"

    # risk, part 2: each pair, counted twice, two pairs per line
    pair_terms = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            pair_terms.append(r"2 \times (%.5f) \; x_{\text{%s}} \, x_{\text{%s}}" % (cov_matrix.loc[names[i], names[j]], names[i], names[j]))
    pairs = r"\begin{array}{rlrl}"
    for k in range(0, len(pair_terms), 2):
        sign = "+"
        if k == 0:
            sign = ""
        pairs += sign + " & " + pair_terms[k]
        if k + 1 < len(pair_terms):
            pairs += r" & \quad + & " + pair_terms[k + 1]
        pairs += r" \\[6pt]"
    pairs += r"\end{array}"

    display(Markdown("**The covariance matrix** (every number in the risk comes from here):"))
    display(cov_matrix.round(5))
    display(Markdown("**Objective:** make the average monthly return as big as possible"))
    display(Math(objective))
    display(Markdown("**Constraint 1:** invest all of the money"))
    display(Math(budget))
    display(Markdown("**Constraint 2:** the risk, $\\sum_i \\sum_j x_i\\,x_j\\,\\text{Cov}(i,j)$, stays under the limit. "
                     "Part 1, each stock with itself (the **diagonal** of the matrix):"))
    display(Math(diagonal))
    display(Markdown("Part 2, each pair of stocks (the **off-diagonal** cells; each pair is in the grid twice, so it gets a 2):"))
    display(Math(pairs))
    display(Markdown("Add parts 1 and 2 together, and the total must be **at most %s**. And every share is between 0 and 1: "
                     "$0 \\le x_k \\le 1$." % risk_limit))


show_model(df_return, df_cov, 0.004)

Five stocks give 5 return terms, but **15** risk terms (5 on the diagonal + 10 pairs). Every new stock adds one
term to the return and a whole row of pairs to the risk. That is why we build the risk with a loop.

### A simple example you know: Apple, Microsoft, NVIDIA and Bitcoin

Same model, four assets you have heard of: Apple, Microsoft, NVIDIA, and IBIT (the iShares Bitcoin Trust, a fund
that holds Bitcoin). IBIT only started trading in January 2024, so we have 32 months, February 2024 to September 2026.

In [ ]:
four = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/four_assets_monthly_returns.csv', index_col=0)
print(len(four), "months")
four.head()

In [ ]:
# the model for these four, at a risk limit of 0.006
show_model(four.mean(), four.cov(), 0.006)
print()
print("correlations (for your eyes):")
print(four.corr().round(2))

Read the numbers before you solve anything:

- NVIDIA has the best average (about 4.7% a month), but IBIT has by far the biggest variance (0.024): Bitcoin swings hard.
- Apple and IBIT have a **negative** covariance: in these months they tended to move in opposite directions. In the risk,
  that pair term *subtracts*, which is what makes mixing them attractive.
- You will solve this exact model in Part 4 (pick "Apple + Microsoft + NVIDIA + Bitcoin" from the menu).

**Look before you model.** Back to Ms. Womack. Here are her 24 months, one line per stock. Messy! Some volatility here.

**Predict first.** If Ms. Womack had to put **all** of her money in one stock, which one would you pick, and why?
Write it down before you scroll on.

In [ ]:
# the raw monthly returns
df.plot(figsize=(10, 4))
axhline(0, color='gray', linewidth=0.8)
xlabel('Month')
ylabel('Monthly return')
title("Ms. Womack's five stocks, month by month")
show()

### What variance measures

Variance is the whole story of risk in this model, so let's see it. Take one stock, draw its average, and look at
how far each month lands from that average. Each of those distances is a **deviation from the mean**,
$r_{k,t} - \bar r_k$ (the red lines in the picture). Variance squares every deviation, then averages them:

$$\text{Var}(k) = \frac{1}{n-1} \sum_{t=1}^{n} \left(r_{k,t} - \bar r_k\right)^2 \qquad \text{(the average squared deviation)}$$

The **standard deviation** (the "volatility" in the table above) is its square root, $\sigma_k = \sqrt{\text{Var}(k)}$,
which puts it back in the units of a return. If monthly returns are roughly bell-shaped (normal), about **2 months in
3** land within $\pm 1\sigma$ of the average and about **19 in 20** within $\pm 2\sigma$. The shaded bands in the
picture show those ranges, and the cell counts how many of the 24 months really landed inside.

**Predict first.** Computer's deviations look about two or three times as long as Power's. How many times bigger is
Computer's **variance**? Write down a number, then run the cell.

In [ ]:
# deviations from the mean (red), with the +/- 1 and 2 standard deviation bands
fig, axes = subplots(1, 2, figsize=(13, 4.8), sharey=True)
pair = ['Power', 'Computer']

k = 0
for stock in pair:
    ax = axes[k]
    avg = df[stock].mean()
    sd = df[stock].std()
    ax.axhspan(avg - 2 * sd, avg + 2 * sd, color='tab:blue', alpha=0.07, label='average +/- 2 std. dev.')
    ax.axhspan(avg - sd, avg + sd, color='tab:blue', alpha=0.15, label='average +/- 1 std. dev.')
    for t in range(len(df)):
        ax.plot([t, t], [avg, df[stock].iloc[t]], color='tab:red', linewidth=2, alpha=0.6)
    ax.plot(range(len(df)), df[stock], 'o', color='black')
    ax.axhline(avg, color='tab:blue', linewidth=2, label='average')
    ax.set_title('%s: std. dev. = %.4f, variance = %.5f' % (stock, sd, df[stock].var()))
    ax.set_xlabel('Month')
    ax.legend(fontsize=8, loc='lower left')
    k += 1
axes[0].set_ylabel('Monthly return')
tight_layout()
show()

for stock in pair:
    avg = df[stock].mean()
    sd = df[stock].std()
    inside_1 = 0
    inside_2 = 0
    for r in df[stock]:
        if abs(r - avg) <= sd:
            inside_1 += 1
        if abs(r - avg) <= 2 * sd:
            inside_2 += 1
    print("%-8s most months between %+.1f%% and %+.1f%%:  %d of 24 within 1 std. dev. (normal: 16),  %d of 24 within 2 (normal: 23)"
          % (stock, 100 * (avg - sd), 100 * (avg + sd), inside_1, inside_2))

### The numbers behind the picture: big swings dominate

Squaring is the point. A deviation 10 times bigger counts **100 times** more:

| deviation | squared |
|---|---|
| 0.02 (a 2% miss) | 0.0004 |
| 0.20 (a 20% miss) | 0.0400 |

Here are each stock's five biggest months, with each deviation, its square, and its share of the stock's total.

In [ ]:
# each month's deviation, squared: which months make up the variance?
for stock in pair:
    deviation = df[stock] - df[stock].mean()
    squared = deviation ** 2
    table = pd.DataFrame({'return': df[stock], 'deviation (r - average)': deviation, 'squared deviation': squared})
    table['share of the total'] = squared / squared.sum()
    table = table.sort_values('squared deviation', ascending=False)
    top5_share = table['share of the total'].head(5).sum()
    print("%s: the 5 biggest months (of 24) make up %.0f%% of its variance" % (stock, 100 * top5_share))
    display(table.head(5).round(4))

sd_ratio = df['Computer'].std() / df['Power'].std()
var_ratio = df['Computer'].var() / df['Power'].var()
print("Computer's deviations are %.1f times as big as Power's (std. dev.)" % sd_ratio)
print("...so its variance is %.1f times as big (%.1f squared)" % (var_ratio, sd_ratio))
print()
for stock in pair:
    print("$10,000 in %-8s: the worst month cost $%s" % (stock, "{:,.0f}".format(-10000 * df[stock].min())))

A handful of big months make up most of the variance. Because every deviation is squared, a stock whose swings
are 2.7 times wider is more than **7 times** riskier in variance, the same way squared error punished big misses in
4.1. That is what Ms. Womack feels: not the average, but the bad months.

### What `calc_risk` adds up

`calc_risk` from M4.2 builds a grid the same shape as the covariance matrix. Each cell is

$$x_i \times x_j \times \text{Cov}(i,j)$$

and **risk is the sum of every cell in the grid**. Here it is for two stocks:

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/calc_risk_two_stocks.png" width="75%"/>

Ms. Womack has five stocks, so her grid has 5 x 5 = 25 cells: 5 on the diagonal (each stock's own variance) and 20
off the diagonal (10 pairs, each one twice). Here it is with her real covariances. Green cells are negative: those
pairs tend to move in opposite directions, and they **subtract** risk.

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/calc_risk_womack_5x5.png" width="100%"/>

Now let's weight the grid for two different portfolios: an **even split** (20% in each) and a **concentrated bet**
(80% in Auto, 5% in each of the others). Same five stocks, same covariance matrix; only the weights change.

**Why Auto?** It has the highest average monthly return of the five (about 2.26%, just ahead of Computer's 2.09%),
so it is the stock a return-chaser would pile into. The question is what that costs in risk.

**Predict first.** Will the concentrated bet be riskier than the even split? By how much: 1.5 times, 3 times, 10
times? Write your guess down, then run the next two cells.

In [ ]:
# calc_risk, written for any set of weights: the weighted covariance grid, then its sum
def weighted_grid(weights, matrix):
    grid = pd.DataFrame(0.0, index=matrix.index, columns=matrix.columns)
    for i in matrix.index:
        for j in matrix.columns:
            grid.loc[i, j] = weights[i] * weights[j] * matrix.loc[i, j]
    return grid


def calc_risk(weights, matrix):
    return weighted_grid(weights, matrix).values.sum()


even_split = {'Computer': 0.2, 'Chemical': 0.2, 'Power': 0.2, 'Auto': 0.2, 'Electronic': 0.2}
all_in_auto = {'Computer': 0.05, 'Chemical': 0.05, 'Power': 0.05, 'Auto': 0.8, 'Electronic': 0.05}

print("risk of the even split:       %.6f" % calc_risk(even_split, df_cov))
print("risk of the concentrated bet: %.6f" % calc_risk(all_in_auto, df_cov))
print("the concentrated bet is %.1f times riskier. What was your guess?" % (calc_risk(all_in_auto, df_cov) / calc_risk(even_split, df_cov)))

In [ ]:
# the raw covariance matrix, then the two weighted grids: risk is the sum of all 25 cells
fig, axes = subplots(1, 3, figsize=(20, 6.5))
panels = {'Raw covariance matrix': None, 'Even split (20% each)': even_split, '80% in Auto': all_in_auto}

k = 0
for name in panels:
    ax = axes[k]
    if panels[name] is None:
        shown = df_cov.values * 10000          # the cells are tiny, so we show them x 10,000
        labels = df_cov
        ax.imshow(shown, cmap='Blues', vmin=-10, vmax=100)
        ax.set_title('Raw covariance matrix (x 10,000)\nthe same for every portfolio')
    else:
        grid = weighted_grid(panels[name], df_cov)
        shown = grid.values * 10000
        labels = grid
        ax.imshow(shown, cmap='Reds', vmin=0, vmax=45)
        ax.set_title(name + '\neach cell = raw cell x share i x share j\ncells add up to %.1f, so risk = %.4f' % (shown.sum(), grid.values.sum()))
    ax.set_xticks(range(len(labels.columns)))
    ax.set_xticklabels(labels.columns, rotation=45)
    ax.set_yticks(range(len(labels.index)))
    ax.set_yticklabels(labels.index)
    for r in range(len(labels.index)):
        for c in range(len(labels.columns)):
            text_color = 'black'
            if (panels[name] is None and shown[r, c] > 60) or (panels[name] is not None and shown[r, c] > 25):
                text_color = 'white'      # readable on the darkest cells
            ax.text(c, r, '%.1f' % shown[r, c], ha='center', va='center', fontsize=10, color=text_color)
    k += 1

tight_layout()
show()

**Read left to right** (every cell is shown x 10,000 so you can read it). The raw covariance matrix never
changes; the weights do. Each weighted cell is the raw cell times the two shares:

- Even split: every pair of shares is $0.2 \times 0.2 = 0.04$, so Computer's raw **96.2** becomes $96.2 \times 0.04 = 3.8$.
- 80% in Auto: Auto with itself is $0.8 \times 0.8 = 0.64$, so Auto's raw **68.8** becomes $68.8 \times 0.64 \approx 44$,
  while Computer's 96.2 shrinks to $96.2 \times 0.05 \times 0.05 = 0.2$.

In the even split, the risk is spread across many small cells. In the concentrated bet, one cell (Auto with itself)
dominates the whole grid, and the total is about **3.4 times bigger**. Holding stocks that do not move together keeps
the off-diagonal cells small. That is diversification, in one picture.

# Part 2 · Why covariance and not correlation?

Both matrices describe how stocks move together, so why does the model use covariance? Here is the whole difference
in one line:

$$\text{Corr}(i,j) = \frac{\text{Cov}(i,j)}{\sigma_i \, \sigma_j}$$

Correlation is covariance with **the size of each stock's swings divided out**. It keeps the *direction* (do they
move together?) and throws away the *size* (how much?). One test shows what that costs.

### The all-in test

Put 100% of the money in one stock. Its risk should be that stock's own variance: Computer (the wildest) should come
out far riskier than Power (the calmest). Here is what `calc_risk` says with each matrix.

In [ ]:
df_corr = df.corr()

# all of the money in one stock, one stock at a time
all_in_cov = {}
all_in_corr = {}
for stock in df.columns:
    all_in = {}
    for s in df.columns:
        all_in[s] = 0.0
    all_in[stock] = 1.0
    all_in_cov[stock] = calc_risk(all_in, df_cov)
    all_in_corr[stock] = calc_risk(all_in, df_corr)

fig, axes = subplots(1, 2, figsize=(13, 4))
axes[0].bar(list(all_in_cov.keys()), list(all_in_cov.values()), color='tab:red')
axes[0].set_title('With COVARIANCE: Computer is %.1f times riskier than Power' % (all_in_cov['Computer'] / all_in_cov['Power']))
axes[0].set_ylabel('risk of going all-in')
axes[1].bar(list(all_in_corr.keys()), list(all_in_corr.values()), color='gray')
axes[1].set_title('With CORRELATION: every stock has risk 1.0')
tight_layout()
show()

Correlation's diagonal is always 1, so an optimizer fed correlation would think going all-in on Computer is
exactly as safe as going all-in on Power. A stock's own swings have disappeared from the model; only the
correlations between pairs are left.

### What that does to Ms. Womack's portfolio

Ask each matrix the same question: **what is the safest way to split her money?** Then check how much each answer
really bounces, using the real risk (covariance), in dollars on \$10,000.

In [ ]:
# the safest possible mix (smallest risk, all of the money invested) according to one matrix
def safest_mix(matrix):
    stocks = list(matrix.index)
    model = ConcreteModel()
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)
    model.constraints = ConstraintList()
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * matrix.loc[i, j] * model.x[j]
    model.risk = Objective(expr=risk_expr, sense=minimize)
    SolverFactory('ipopt', executable=ipopt_path).solve(model)
    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return weights


by_cov = safest_mix(df_cov)
by_corr = safest_mix(df_corr)

side_by_side = pd.DataFrame({"covariance's answer": by_cov, "correlation's answer": by_corr})
bounce_cov = 10000 * np.sqrt(calc_risk(by_cov, df_cov))
bounce_corr = 10000 * np.sqrt(calc_risk(by_corr, df_cov))

fig, axes = subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [2, 1]})
side_by_side.plot(kind='bar', ax=axes[0], color=['tab:red', 'gray'], rot=0)
axes[0].set_ylabel('Share of the money')
axes[0].set_title('"The safest mix", according to each matrix')

axes[1].bar(["covariance's\nanswer", "correlation's\nanswer"], [bounce_cov, bounce_corr], color=['tab:red', 'gray'], width=0.6)
axes[1].text(0, bounce_cov + 8, '$%.0f' % bounce_cov, ha='center', fontsize=16, fontweight='bold')
axes[1].text(1, bounce_corr + 8, '$%.0f' % bounce_corr, ha='center', fontsize=16, fontweight='bold')
axes[1].set_ylim(0, bounce_corr * 1.25)
axes[1].set_ylabel(r'Real monthly bounce on \$10,000 (\$)')
axes[1].set_title('Correlation\'s "safest" mix really\nbounces %.0f%% more' % (100 * (bounce_corr / bounce_cov - 1)), fontweight='bold')
tight_layout()
show()

<div style="border-left: 8px solid #c92a2a; background: #fff1f1; color: #1a1a1a; padding: 14px 18px; font-size: 17px; line-height: 1.5;">
<b>The bottom line.</b> Same data, same question, two matrices, <b>two different portfolios</b>. Correlation thinks
every stock swings the same, so its "safest" mix spreads the money almost evenly and puts a fifth of it in Computer,
the wildest stock. It really bounces about <b>&#36;337 a month</b> on &#36;10,000, against <b>&#36;259</b> for covariance's
answer, which leans on calm Power.<br><br>
Fed correlation, the optimizer hands Ms. Womack a riskier portfolio and calls it the safe one.
<b>Always give the optimizer covariance.</b>
</div>

Here are the two matrices side by side. Look at the diagonals:

In [ ]:
# the two matrices, side by side
fig, axes = subplots(1, 2, figsize=(13, 5))
matrices = {'Covariance (what the optimizer needs)': df_cov, 'Correlation (what your eyes like)': df_corr}

k = 0
for name in matrices:
    m = matrices[name]
    ax = axes[k]
    ax.imshow(m.values, cmap='coolwarm')
    ax.set_xticks(range(len(m.columns)))
    ax.set_xticklabels(m.columns, rotation=45)
    ax.set_yticks(range(len(m.index)))
    ax.set_yticklabels(m.index)
    for r in range(len(m.index)):
        for c in range(len(m.columns)):
            ax.text(c, r, '%.4f' % m.values[r, c], ha='center', va='center', fontsize=8)
    ax.set_title(name)
    k += 1

tight_layout()
show()

### The takeaway

- **Correlation's diagonal is always 1.0.** Power (the calmest stock) and Computer (the wildest) both get a 1 on the
  diagonal, so correlation throws away **how much each stock moves on its own**. Correlation only keeps the
  *direction* two stocks move together, on a scale from -1 to 1.
- **Covariance keeps the size.** Its diagonal is each stock's own variance (Computer 0.0096 vs. Power 0.0013), and
  each off-diagonal cell is in the same units, so adding them up gives the variance of the whole portfolio.

In dollars: for the even split on \$10,000, covariance predicts a typical monthly bounce of about **\$374**, which is
exactly what her real 24 months did. The same formula with correlation predicts about \$5,343, more than half of her
money, every month. That is not the risk of anything.

> **Correlation is for your eyes; covariance is for the optimizer.** Use correlation to *see* which stocks move
> together. Feed covariance to the model.

### Bounce, for any portfolio

Risk in this model is a variance, and variance is hard to feel. Its square root is easier: the portfolio's
**bounce** (its standard deviation),

$$\text{bounce} = \sigma_{\text{portfolio}} = \sqrt{\sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j)} = \sqrt{\texttt{calc\_risk}}$$

On \$10,000 it reads in dollars. Each month the portfolio earns its average month **plus or minus** some amount.
If the months are roughly bell-shaped, about **2 months in 3** land within one bounce of the average and about
**19 in 20** within two bounces. So a bounce of \$374 means: most months land within \$374 of the average month,
and a month more than \$748 below it should be rare.

The best part: `calc_risk` predicts the bounce **from the covariance matrix alone**, before you invest a dollar. Type
in any mix and check the prediction against what really happened in the 24 months. Try the even split, then 80% in
Auto, then all of the money in Computer, then all of it in Power.

In [ ]:
#@title Bounce check (shares in percent; they must add up to 100)
computer_pct = 20  #@param {type:"number"}
chemical_pct = 20  #@param {type:"number"}
power_pct = 20  #@param {type:"number"}
auto_pct = 20  #@param {type:"number"}
electronic_pct = 20  #@param {type:"number"}

mix = {'Computer': computer_pct / 100, 'Chemical': chemical_pct / 100, 'Power': power_pct / 100, 'Auto': auto_pct / 100, 'Electronic': electronic_pct / 100}
total = 0
for s in mix:
    total += mix[s]

if abs(total - 1) > 0.0001:
    print("Your shares add up to %.0f%%, not 100%%. Fix them and run again." % (100 * total))
else:
    # the prediction, from the covariance matrix alone
    average_month = 0
    for s in mix:
        average_month += 10000 * mix[s] * df_return[s]
    bounce = 10000 * np.sqrt(calc_risk(mix, df_cov))

    # what really happened, month by month, in dollars
    dollars = 0
    for s in mix:
        dollars = dollars + 10000 * mix[s] * df[s]

    figure(figsize=(10, 4))
    axhspan(average_month - 2 * bounce, average_month + 2 * bounce, color='tab:blue', alpha=0.08, label='average +/- 2 bounces')
    axhspan(average_month - bounce, average_month + bounce, color='tab:blue', alpha=0.18, label='average +/- 1 bounce')
    axhline(average_month, color='tab:blue', linewidth=2, label='average month')
    bar(range(len(dollars)), dollars, color='black', width=0.6, label='what really happened')
    axhline(0, color='gray', linewidth=0.8)
    ylim(-2500, 3000)        # the same scale for every mix, so you can compare
    xlabel('Month')
    ylabel(r'Gain or loss on \$10,000 (\$)')
    title('Predicted bounce: +/- $%s a month' % "{:,.0f}".format(bounce))
    legend(fontsize=8, loc='upper right')
    show()

    inside_1 = 0
    inside_2 = 0
    for d in dollars:
        if abs(d - average_month) <= bounce:
            inside_1 += 1
        if abs(d - average_month) <= 2 * bounce:
            inside_2 += 1
    print("average month: $%s    bounce: +/- $%s" % ("{:,.0f}".format(average_month), "{:,.0f}".format(bounce)))
    print("months within 1 bounce: %d of 24 (a bell curve says about 16)" % inside_1)
    print("months within 2 bounces: %d of 24 (a bell curve says about 23)" % inside_2)
    print("worst month: %s    best month: +$%s" % ("{:,.0f}".format(dollars.min()).replace("-", "-$"), "{:,.0f}".format(dollars.max())))

The model never limits the *average*: it limits the bounce (squared). Every risk limit you set is really a
sentence Ms. Womack can understand: "I can live with my \$10,000 bouncing about this many dollars a month." A limit
of 0.004, for example, is a bounce of $\sqrt{0.004} \approx 0.063$, or about **\$632 a month** on \$10,000.

# Part 3 · Ms. Womack's efficient frontier

The model from Part 1, written once so it works for **any** set of stocks: the stock names come from the data, and
loops build the return and the risk (the same `+=` pattern as Studio 3). `solve_portfolio(risk_limit)` answers one
question: *what is the best average return without the risk going over `risk_limit`?*

In [ ]:
import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)    # keeps the output tidy when a risk limit is too low to reach


def solve_portfolio(risk_limit):
    stocks = list(df_return.index)
    model = ConcreteModel()

    # the share of the money in each stock, between 0 and 1 (start from an even split)
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # risk: the weighted covariance matrix, summed (calc_risk), under the limit
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
    if results.solver.termination_condition != TerminationCondition.optimal:
        return None      # no portfolio is that safe

    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights}


print("solve_portfolio is ready")

**Check it against M4.2.** Same data, same model, so we must get the same answers as the original notebook:

| risk limit | best return in M4.2 |
|---|---|
| 0.002 | 0.0175 (about 1.75% a month) |
| 0.004 | 0.0208 |
| 0.006 | 0.0224 |

In [ ]:
# does our model match M4.2?
expected = {0.002: 0.017542, 0.004: 0.020786, 0.006: 0.022374}

for risk_limit in expected:
    answer = solve_portfolio(risk_limit)
    if abs(answer['return'] - expected[risk_limit]) < 0.00005:
        print("risk limit", risk_limit, ": best return = %.4f   matches M4.2" % answer['return'])
    else:
        print("risk limit", risk_limit, ": best return = %.4f   does NOT match M4.2" % answer['return'])

### Beat the solver: the contest

Before the solver draws the whole frontier, you try to beat it by hand. The rules:

- Ms. Womack will accept a risk of **at most 0.004**. Over the limit, the try does not count.
- Type her five shares as **percentages** (they must add up to 100) and run the cell. You get the return and the
  risk of your mix, plus your best valid return so far. Every try is saved.
- You have about five minutes. When time is up, **write your best return on the board**.
- Do not run the reveal cell until Dr. Dave says so.

Run this cell once to start the contest (running it again erases your tries).

In [ ]:
# every try you make is saved here
my_tries = []
womack_limit = 0.004
print("Contest on: risk limit", womack_limit)

In [ ]:
#@title Your mix for Ms. Womack, in percent (risk limit 0.004)
Computer = 20  #@param {type:"number"}
Chemical = 20  #@param {type:"number"}
Power = 20  #@param {type:"number"}
Auto = 20  #@param {type:"number"}
Electronic = 20  #@param {type:"number"}

my_mix = {'Computer': Computer / 100, 'Chemical': Chemical / 100, 'Power': Power / 100, 'Auto': Auto / 100, 'Electronic': Electronic / 100}

total = 0
for s in my_mix:
    total += my_mix[s]

if abs(total - 1) > 0.0001:
    print("Your shares add up to %.0f%%, not 100%%. Fix them and run again." % (100 * total))
else:
    my_risk = calc_risk(my_mix, df_cov)
    my_return = 0
    for s in my_mix:
        my_return += df_return[s] * my_mix[s]
    within = my_risk <= womack_limit
    my_tries.append({'return': my_return, 'risk': my_risk, 'within limit': within})

    print("Try %d:  return %.4f  (%.2f%% a month)   risk %.5f" % (len(my_tries), my_return, 100 * my_return, my_risk))
    if within:
        print("        within the limit: this one counts")
    else:
        print("        OVER the limit of %.3f: Ms. Womack says no" % womack_limit)

    best_so_far = None
    for t in my_tries:
        if t['within limit']:
            if best_so_far is None or t['return'] > best_so_far:
                best_so_far = t['return']
    if best_so_far is not None:
        print("Your best valid return so far: %.4f" % best_so_far)

**Reveal (only when Dr. Dave says so).** The solver's best mix at the same limit, and how close you got.

In [ ]:
best = solve_portfolio(womack_limit)

print("THE SOLVER:  return %.4f  (%.2f%% a month)   risk %.5f" % (best['return'], 100 * best['return'], best['risk']))
for s in best['weights']:
    print("   %-10s %5.1f%%" % (s, 100 * best['weights'][s]))

best_so_far = None
for t in my_tries:
    if t['within limit']:
        if best_so_far is None or t['return'] > best_so_far:
            best_so_far = t['return']
if best_so_far is not None:
    print()
    print("Your best: %.4f, which is %.1f%% of the solver's return" % (best_so_far, 100 * best_so_far / best['return']))

Look at the solver's mix. Which stock did it lean on, and which did it leave out entirely? Was the lean stock the
one you would have gone all-in on at the start? Hold on to your tries: they show up on the frontier in a moment.

### Solve it at many risk limits

Each solve answers one question ("given this much risk, what is the best return?"). Stringing the answers together
draws the **efficient frontier**. The range of risk limits comes from the data: from very safe up to the risk of the
single stock with the highest average return (past that, more risk buys nothing).

In [ ]:
# the risk limits, from the data: the top is the variance of the best-returning stock
def frontier(n_levels):
    best_stock = df_return.idxmax()
    top = df_cov.loc[best_stock, best_stock]
    bottom = df_cov.values.diagonal().min() / 10
    risk_limits = np.linspace(bottom, top, n_levels)

    param_analysis = {}      # risk limit -> what to hold
    returns = {}             # risk limit -> the best return
    for r in risk_limits:
        answer = solve_portfolio(r)
        if answer is None:
            continue         # too safe to reach: skip it
        param_analysis[r] = answer['weights']
        returns[r] = answer['return']
    return pd.DataFrame(param_analysis).T, pd.Series(returns)


param_analysis, returns = frontier(40)
print(len(returns), "portfolios on the frontier")

In [ ]:
# the efficient frontier, with each single stock for comparison
def draw_frontier(chosen_risk=None, chosen_return=None, benchmark=None, tries=None):
    figure(figsize=(9, 5.5))
    plot(returns.index, returns.values, marker='o', markersize=4, label='efficient frontier')
    if tries is not None:
        for t in tries:
            if t['within limit']:
                scatter(t['risk'], t['return'], color='orange', edgecolor='black', s=60, zorder=4)
            else:
                scatter(t['risk'], t['return'], color='orange', marker='x', s=60, zorder=4)
        scatter([], [], color='orange', edgecolor='black', s=60, label='your tries (x = over the limit)')
    for s in df_return.index:
        scatter(df_cov.loc[s, s], df_return[s], color='gray')
        annotate(s, (df_cov.loc[s, s], df_return[s]), fontsize=8)
    if benchmark is not None:
        scatter(benchmark.var(), benchmark.mean(), color='red', marker='*', s=250, label='S&P 500 (SPY)')
    if chosen_risk is not None:
        axvline(chosen_risk, color='green', linestyle='--', label='risk limit')
        scatter(chosen_risk, chosen_return, color='green', s=150, zorder=5)
    xlabel('Risk (variance of the monthly return)')
    ylabel('Average monthly return')
    title('The efficient frontier')
    legend(fontsize=8)
    show()


# Ms. Womack's frontier, her limit of 0.004, and every one of your contest tries
draw_frontier(chosen_risk=womack_limit, chosen_return=best['return'], tries=my_tries)

**Every dot is the best return possible under one risk limit.** No portfolio can sit above the curve, and every
single stock sits below it: some mix of the five always does at least as well as any one of them. Your contest tries
(orange) all sit on or below the curve too, and the best of them sit closest to where the green line meets it.

### Diminishing returns: what does one more step of risk buy?

The frontier bends. Here is what that bend means in money. Raise Ms. Womack's risk limit in **equal steps** of
0.0005 and re-solve the model each time (the same move as the shadow prices in Studio 3: what is one more hour
worth?). Each green bar is what that one step buys:

$$\text{step } k \text{ buys} = \$100{,}000 \times \Big( \text{best return at limit}_k - \text{best return at limit}_{k-1} \Big) \text{ a month}$$

For example, the best return is 1.36% a month at a limit of 0.0010 and 1.59% at 0.0015, so that step buys
$0.0023 \times \$100{,}000 \approx \$229$ a month.

**Where to stop?** Ms. Womack decides how much one more step of risk has to pay her, say \$50 a month. She keeps
raising the limit while each step pays at least that much and stops at the first step that pays less. That is her
**best value**. On the chart, her price is the dashed line across the bars (bars below it turn gray), and her best value is the dashed line on the frontier. Move the slider to change her price.

**Predict first.** Every step adds the same amount of risk. Does every step buy the same amount of return? Write
down yes or no, then run the cell.

In [ ]:
# Ms. Womack's price for risk: what one more step must pay her, in dollars a month
price_of_risk = 50  #@param {type:"slider", min:0, max:250, step:10}

# raise the risk limit in equal steps and see what each step buys
step = 0.0005
savings = 100000                      # Ms. Womack's savings, in dollars
levels = np.arange(0.001, 0.0075 + step / 2, step)

best_at = {}
for r in levels:
    answer = solve_portfolio(r)
    if answer is not None:
        best_at[r] = answer['return']
limits = list(best_at.keys())

gains = []
step_labels = []
for k in range(1, len(limits)):
    gains.append(max(0.0, savings * (best_at[limits[k]] - best_at[limits[k - 1]])))      # never below zero
    step_labels.append('%.4f' % limits[k])

# her best value: keep stepping while the step pays at least her price
best_value = limits[0]
for k in range(len(gains)):
    if gains[k] >= price_of_risk:
        best_value = limits[k + 1]
    else:
        break

bar_colors = []
for g in gains:
    if g >= price_of_risk:
        bar_colors.append('tab:green')
    else:
        bar_colors.append('lightgray')

fig, axes = subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.plot(returns.index, returns.values, color='lightgray', linewidth=6, zorder=1, label='efficient frontier')
for k in range(1, len(limits)):
    ax.plot([limits[k - 1], limits[k]], [best_at[limits[k - 1]], best_at[limits[k - 1]]], color='tab:blue', linewidth=2)
    ax.plot([limits[k], limits[k]], [best_at[limits[k - 1]], best_at[limits[k]]], color='tab:green', linewidth=3)
ax.axvline(best_value, color='black', linestyle='--', label='her best value (%.4f)' % best_value)
ax.scatter(best_value, best_at[best_value], color='black', s=80, zorder=5)
ax.set_xlabel('Risk limit')
ax.set_ylabel('Best average monthly return')
ax.set_title('Equal steps of risk (blue), the return each one buys (green)')
ax.legend(fontsize=8)

axes[1].bar(range(len(gains)), gains, color=bar_colors)
axes[1].axhline(price_of_risk, color='black', linestyle='--', label='her price: $%d a month' % price_of_risk)
axes[1].set_xticks(range(len(gains)))
axes[1].set_xticklabels(step_labels, rotation=45)
axes[1].set_xlabel('Risk limit raised to')
axes[1].set_ylabel(r'Extra return per month on \$100,000 (\$)')
axes[1].set_title('What each extra 0.0005 of risk buys')
axes[1].legend(fontsize=8)
tight_layout()
show()

for k in range(len(gains)):
    print("risk limit %s: + $%5.0f a month" % (step_labels[k], gains[k]))
print()
print("Her best value at $%d a month: risk limit %.4f, best return %.2f%% a month ($%s a month on $100,000)"
      % (price_of_risk, best_value, 100 * best_at[best_value], "{:,.0f}".format(savings * best_at[best_value])))

**Diminishing returns.** The first steps of risk buy a lot of return; each later step buys less; and past
Auto's own variance (about 0.0069) a step buys **nothing**, because the money is already all in Auto and there is no
higher-returning stock left to move into. A pickier investor (a higher price) stops sooner, at a safer portfolio; a
less picky one keeps going. Either way, the bend in the frontier is what makes stopping somewhere sensible.

In [ ]:
# what Ms. Womack holds at each risk limit
stackplot(param_analysis.index, param_analysis.T.values, labels=param_analysis.columns)
xlabel('Risk (variance of the monthly return)')
ylabel('Share of the money')
title('What to hold along the frontier')
legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
show()

On the left (low risk) the money is spread across many stocks. On the right (high risk) it piles into the stock
with the highest average return. As M4.2 put it: **look for the top-left corner** of the frontier, where you still
get most of the return without taking on all of the risk.

# Part 4 · Real stocks

Same model, same code, real data.

### Where do monthly returns come from?

Ms. Womack's file arrived as monthly **returns**. Real data arrives as month-end **prices**, and the model needs
returns, so every real-data project starts with this step:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

Watch it happen for one Apple share, one month at a time. Each month's return compares two prices: this month's and
last month's. The first month has no return, because there is no month before it.

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/prices_to_returns.gif" width="80%"/>

In pandas this is one line, `prices.pct_change().dropna()`. The 10-company file below has already been through it
(Studio 4.3 does it from scratch, including a trap to watch out for).

### Pick a data set

Pick a data set from the menu and run the cell. Everything after it updates on its own: click on the cell and
choose **Runtime → Run cell and below**.

- **10 large companies** is ready to go: one from each of ten sectors (Apple, JPMorgan Chase, Johnson & Johnson,
  ExxonMobil, Coca-Cola, Home Depot, Caterpillar, NextEra Energy, Alphabet, Linde), 60 months from October 2021 to
  September 2026.
- **Apple + Microsoft + NVIDIA + Bitcoin** is the four-asset example from Part 1 (32 months, February 2024 to
  September 2026).
- **My own tickers** downloads monthly returns from Yahoo Finance for whatever you type in. If the download fails
  (it sometimes does in Colab), the cell falls back to the 10 companies.

In [ ]:
#@title Pick the stocks
data_choice = "10 large companies, one per sector"  #@param ["10 large companies, one per sector", "Apple + Microsoft + NVIDIA + Bitcoin", "My own tickers"]
my_tickers = "AAPL, MSFT, NVDA, KO, XOM, JPM"  #@param {type:"string"}

data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/'
df = pd.read_csv(data_folder + 'stocks_monthly_returns.csv', index_col=0)

if data_choice == "My own tickers":
    try:
        import yfinance as yf
        tickers = []
        for t in my_tickers.split(','):
            tickers.append(t.strip().upper())
        prices = yf.download(tickers, start='2021-09-01', end='2026-10-01', interval='1mo', auto_adjust=True, progress=False)['Close']
        downloaded = prices.pct_change().dropna()
        if len(downloaded) > 12:
            df = downloaded
            print("downloaded", len(df), "months for", list(df.columns))
        else:
            print("The download came back too short, so we are using the 10 companies instead.")
    except Exception as problem:
        print("The download did not work (", problem, "), so we are using the 10 companies instead.")

spy = pd.read_csv(data_folder + 'spy_monthly_returns.csv', index_col=0)['SPY']

if data_choice == "Apple + Microsoft + NVIDIA + Bitcoin":
    df = pd.read_csv(data_folder + 'four_assets_monthly_returns.csv', index_col=0)
    spy = spy.loc[df.index]          # compare with the S&P 500 over the same 32 months

df_return = df.mean()
df_cov = df.cov()
print(len(df), "months,", len(df.columns), "stocks:", list(df.columns))
pd.DataFrame({'average monthly return': df_return, 'volatility (std. dev.)': df.std()}).round(4)

In [ ]:
# the same model, the same loop, new data
param_analysis, returns = frontier(40)
print(len(returns), "portfolios on the frontier")
draw_frontier(benchmark=spy)

In [ ]:
# what to hold along the frontier
stackplot(param_analysis.index, param_analysis.T.values, labels=param_analysis.columns)
xlabel('Risk (variance of the monthly return)')
ylabel('Share of the money')
title('What to hold along the frontier')
legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
show()

### Slide your risk tolerance

0 is the safest portfolio on the frontier, 1 is all-in on the highest-return stock. Move the slider, then run the
cell. The green line is your risk limit; the bars show what to hold.

In [ ]:
#@title Your risk tolerance
risk_tolerance = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}

safest = returns.index.min()
riskiest = returns.index.max()
my_limit = safest + risk_tolerance * (riskiest - safest)
mine = solve_portfolio(my_limit)

print("risk limit %.5f -> average monthly return %.2f%%, monthly volatility %.2f%%"
      % (my_limit, 100 * mine['return'], 100 * np.sqrt(mine['risk'])))
draw_frontier(chosen_risk=mine['risk'], chosen_return=mine['return'], benchmark=spy)

holdings = pd.Series(mine['weights']).sort_values()
barh(holdings.index, holdings.values)
xlabel('Share of the money')
title('What to hold at risk tolerance %.2f' % risk_tolerance)
show()

### Your turn

Answer in a text cell:

1. In Part 2, what risk did correlation give for going all-in on Computer versus all-in on Power? Why is that a problem for an optimizer?
2. On the 10 real companies, which stock does the frontier end on (all the money in one stock)? Would you put all of
   anyone's savings there?
3. Slide the risk tolerance to 0. How many stocks does the safest portfolio hold, and what is its monthly volatility?
4. Where is the S&P 500 star compared with the frontier? Why shouldn't you trust that comparison too much? (Hint:
   these are averages of the past five years, on ten companies picked today.)
5. Pick a risk tolerance for Ms. Womack and explain your choice in two sentences.

## Bottom line

- **Return** is a weighted average (linear). **Risk** is the weighted covariance matrix, summed (nonlinear).
- **Covariance**, not correlation, because only covariance adds up to the real variance of the portfolio.
- Solve "best return under this risk limit" at many limits and you get the **efficient frontier**; the investor
  chooses the point.
- The model never names a stock. Swap the data and the same code runs on anything, which is the starting point for
  your final project.

**Next: Studio 4.3** takes this model to real data the way an analyst would: prices to returns, the split trap, and
a sector of your own, tested on twelve months the model never saw.